In [ ]:
def main(datasources, start_date, end_date):

    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    start_ts = pd.to_datetime(start_date)
    end_ts = pd.to_datetime(end_date)

    # 向前扩展历史
    query_start = (
        start_ts - pd.Timedelta(days=60)
    ).strftime("%Y-%m-%d %H:%M:%S")

    query_end = end_ts.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    # ==============================
    # 1. 分钟转日频
    # ==============================

    sql = f"""
    WITH daily AS (

        SELECT

            DATE(date) AS date,
            instrument,

            FIRST(close ORDER BY date) AS open_px,
            LAST(close ORDER BY date) AS close_px,

            MAX(close) AS high_px,
            MIN(close) AS low_px,

            SUM(volume) AS volume,
            SUM(amount) AS amount,

            SUM(amount)
            /
            NULLIF(SUM(volume),0)
            AS vwap_px


        FROM {bar1m}

        WHERE
            close > 0
            AND volume >= 0
            AND amount >= 0


        GROUP BY
            DATE(date),
            instrument
    )


    SELECT *

    FROM daily

    ORDER BY
        instrument,
        date
    """

    daily = dai.query(
        sql,
        filters={
            "date":[
                query_start,
                query_end
            ]
        },
        compression=True,
    ).df()


    # ==============================
    # 2. 股票池
    # ==============================

    pool = dai.query(
        """
        SELECT
            date,
            instrument

        FROM bigalpha_2026_instruments
        """,

        filters={
            "date":[
                start_ts.strftime("%Y-%m-%d"),
                end_ts.strftime("%Y-%m-%d")
            ]
        },

        compression=True,
    ).df()


    pool["date"] = pd.to_datetime(
        pool["date"]
    )


    # ==============================
    # 空数据保护
    # ==============================

    if daily.empty:

        pool["factor"] = 0.0

        return pool[
            [
                "date",
                "instrument",
                "factor"
            ]
        ]


    # ==============================
    # 3. 清洗
    # ==============================

    daily["date"] = pd.to_datetime(
        daily["date"]
    )


    cols = [
        "open_px",
        "close_px",
        "high_px",
        "low_px",
        "volume",
        "amount",
        "vwap_px"
    ]


    for c in cols:

        daily[c] = pd.to_numeric(
            daily[c],
            errors="coerce"
        )


    daily = daily.dropna()


    daily = daily.sort_values(
        [
            "instrument",
            "date"
        ]
    )


    eps = 1e-12


    # ==============================
    # 4. 基础市场变量
    # ==============================


    # 当日收益
    daily["ret"] = (
        daily["close_px"]
        /
        daily["open_px"]
        -1
    )


    # VWAP偏离
    daily["vwap_dev"] = (
        daily["close_px"]
        /
        daily["vwap_px"]
        -1
    )


    # 日内振幅
    daily["range"] = (
        daily["high_px"]
        -
        daily["low_px"]
    )


    daily["range_pct"] = (
        daily["range"]
        /
        daily["vwap_px"]
    )


    # 趋势效率
    daily["efficiency"] = (

        (
            daily["close_px"]
            -
            daily["open_px"]
        ).abs()

        /

        daily["range"].clip(
            lower=eps
        )

    ).clip(
        0,
        1
    )



    # ==============================
    # 5. 成交异常
    # ==============================

    g = daily.groupby(
        "instrument",
        sort=False
    )


    daily["volume_mean20"] = g[
        "volume"
    ].transform(
        lambda x:
        x.shift(1)
        .rolling(
            20,
            min_periods=5
        )
        .median()
    )


    daily["amount_mean20"] = g[
        "amount"
    ].transform(
        lambda x:
        x.shift(1)
        .rolling(
            20,
            min_periods=5
        )
        .median()
    )


    daily["volume_mean20"] = (
        daily["volume_mean20"]
        .fillna(
            daily["volume"]
        )
    )


    daily["amount_mean20"] = (
        daily["amount_mean20"]
        .fillna(
            daily["amount"]
        )
    )


    daily["volume_shock"] = (

        daily["volume"]
        /
        daily["volume_mean20"]

    )


    daily["amount_shock"] = (

        daily["amount"]
        /
        daily["amount_mean20"]

    )



    # ==============================
    # 6. 因子构造
    # ==============================


    # 异常成交强度

    activity = (

        np.log(
            daily["volume_shock"]
            +1e-12
        )

        +

        np.log(
            daily["amount_shock"]
            +1e-12
        )

    ) / 2



    # 核心：

    # 价格偏离越大
    # 成交越异常
    # 越可能产生修复

    raw_factor = (

        -0.55
        *
        daily["vwap_dev"]

        *

        (1 + activity.clip(
            -1,
            3
        ))

    )


    # 非趋势震荡加强反转

    raw_factor *= (

        1
        +
        (
            1
            -
            daily["efficiency"]
        )

    )



    # 大波动风险控制

    raw_factor /= (

        1
        +
        daily["range_pct"].clip(
            0,
            1
        )

    )



    daily["signal"] = raw_factor



    # ==============================
    # 7. 时间平滑
    # ==============================


    daily["signal"] = g[
        "signal"
    ].transform(
        lambda x:

        0.7*x

        +

        0.2*x.shift(1).fillna(
            x
        )

        +

        0.1*x.shift(2).fillna(
            x
        )

    )



    # ==============================
    # 8. 回到测试区间
    # ==============================


    daily = daily[

        (daily["date"] >= start_ts)

        &

        (daily["date"] <= end_ts)

    ]



    # ==============================
    # 9. 股票池合并
    # ==============================


    out = pool.merge(

        daily[
            [
                "date",
                "instrument",
                "signal"
            ]
        ],

        how="left",

        on=[
            "date",
            "instrument"
        ]

    )


    out["signal"] = (

        out["signal"]

        .replace(
            [np.inf,-np.inf],
            np.nan
        )

    )


    median = out.groupby(
        "date"
    )["signal"].transform(
        "median"
    )


    out["signal"] = (
        out["signal"]
        .fillna(median)
        .fillna(0)
    )


    # ==============================
    # 10. 横截面排名
    # ==============================

    out["factor"] = (

        out.groupby(
            "date"
        )["signal"]

        .rank(
            pct=True
        )

        -0.5

    )


    out["factor"] = (

        out["factor"]
        .replace(
            [np.inf,-np.inf],
            0
        )
        .fillna(0)
        .astype(float)

    )


    return out[
        [
            "date",
            "instrument",
            "factor"
        ]
    ].sort_values(
        [
            "date",
            "instrument"
        ]
    ).reset_index(
        drop=True
    )